# COCO Data Loading

This notebook prepares the shared MS COCO 2014 caption dataset for the rest of the project, including the first deliverable. It expects the official COCO 2014 files to be placed under `data/raw/coco/` after downloading them from the COCO API repository and dataset release.

Expected raw layout:
- `data/raw/coco/annotations/captions_train2014.json`
- `data/raw/coco/annotations/captions_val2014.json`
- `data/raw/coco/train2014/`
- `data/raw/coco/val2014/`

## Download step

Use the instructions from https://github.com/cocodataset/cocoapi to download the annotations and images, then place them in the raw folder above. Once the files are present, run the next cells to create the shared tabular outputs.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

def find_project_root(start: Path | None = None) -> Path:
    start = start or Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'src').exists() and (candidate / 'data').exists():
            return candidate
    raise FileNotFoundError('Could not locate the project root.')

In [ ]:
required_paths = {
    'train annotations': RAW_COCO_DIR / 'annotations' / 'captions_train2014.json',
    'val annotations': RAW_COCO_DIR / 'annotations' / 'captions_val2014.json',
    'train images': RAW_COCO_DIR / 'train2014',
    'val images': RAW_COCO_DIR / 'val2014',
}

missing = [name for name, path in required_paths.items() if not path.exists()]
if missing:
    message = '\n'.join(f'- {name}: {required_paths[name]}' for name in missing)
    raise FileNotFoundError(
        'Missing COCO files. Download the dataset with the COCO API instructions and place it here:\n'
        f'{message}'
    )

pd.DataFrame({name: [str(path)] for name, path in required_paths.items()})

In [ ]:
from src.api.coco_handler import COCODataLoader

loader = COCODataLoader(RAW_COCO_DIR)

train_df = loader.load_split('train').assign(split='train')
val_df = loader.load_split('val').assign(split='val')
coco_df = pd.concat([train_df, val_df], ignore_index=True)

train_df.head()

In [ ]:
train_out = INTERIM_COCO_DIR / 'captions_train2014.csv'
val_out = INTERIM_COCO_DIR / 'captions_val2014.csv'
all_out = INTERIM_COCO_DIR / 'captions_all.csv'

def to_shared_table(frame: pd.DataFrame) -> pd.DataFrame:
    table = frame.copy()
    table['image_path'] = table['image_path'].astype(str)
    return table

to_shared_table(train_df).to_csv(train_out, index=False)
to_shared_table(val_df).to_csv(val_out, index=False)
to_shared_table(coco_df).to_csv(all_out, index=False)

summary = pd.DataFrame([
    {'split': 'train', 'rows': len(train_df), 'output': str(train_out)},
    {'split': 'val', 'rows': len(val_df), 'output': str(val_out)},
    {'split': 'all', 'rows': len(coco_df), 'output': str(all_out)},
])

summary

## How other notebooks should use this output

Read the shared CSVs from `data/interim/coco/` instead of re-parsing the raw JSON files. This keeps the rest of the notebooks lightweight and consistent.

In [ ]:
shared_coco = pd.read_csv(INTERIM_COCO_DIR / 'captions_all.csv')
shared_coco.head()